In [1]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" stable-ts soundfile librosa
!pip install -q librosa audioread
!pip install stable-whisper


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.1/189.1 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 80.5 MB/s eta 0:00:00
ERROR: pip's dependency re

In [2]:
import re
import numpy as np
import torch
from datasets import load_dataset, Audio, Dataset
from transformers import WhisperProcessor

MODEL_NAME    = "openai/whisper-medium"
SAMPLING_RATE = 16_000
OUTPUT_DIR    = "/kaggle/working/processed_quran_ayah_words"
MAX_LABEL_LEN = 448
PAD_S         = 0.10            # padding kept around each word boundary
DEVICE        = "cuda"

# Subset control: alignment is the slow part. Start SMALL to validate quality,
# then raise. None = use all rows (long!). A few thousand ayahs across reciters
# already yields tens of thousands of word clips.
MAX_AYAHS = 4000

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language="Arabic", task="transcribe"
)

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [3]:
# Same cleaning as Notebook 1 -- keep all diacritics, strip structural marks.
STRUCTURAL_RE = re.compile(
    "[ۖ-ۜ۝۞۩ؐ-ؗـ٠-٩۰-۹0-9]"
)
def clean_quranic_text(text: str) -> str:
    text = STRUCTURAL_RE.sub("", text)
    return re.sub(r"\s+", " ", text).strip()

In [4]:
# 30-reciter verse recordings. Each row: full-verse audio + ayah_ar text +
# reciter/surah ids. We will align each verse and cut it into word clips.
raw = load_dataset("Buraaq/quran-md-ayahs")
ds = raw["train"]
print(ds)
print("columns:", ds.column_names)
print("sample ayah_ar:", ds[0]["ayah_ar"])

# Resolve the reciter column defensively (name may vary across revisions).
RECITER_COL = next((c for c in ["reciter","reciter_id","speaker","speaker_id"]
                    if c in ds.column_names), None)
print("reciter column:", RECITER_COL)
assert RECITER_COL is not None, "No reciter column found -- check ds.column_names"

ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE, mono=True))
if MAX_AYAHS is not None and len(ds) > MAX_AYAHS:
    ds = ds.shuffle(seed=0).select(range(MAX_AYAHS))
print("ayahs to process:", len(ds))

Resolving data files:   0%|          | 0/71 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/71 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/187080 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/63 [00:00<?, ?it/s]

Dataset({
    features: ['surah_id', 'ayah_id', 'surah_name_ar', 'surah_name_en', 'surah_name_tr', 'ayah_count', 'ayah_ar', 'ayah_en', 'ayah_tr', 'reciter_id', 'reciter_name', 'audio'],
    num_rows: 187080
})
columns: ['surah_id', 'ayah_id', 'surah_name_ar', 'surah_name_en', 'surah_name_tr', 'ayah_count', 'ayah_ar', 'ayah_en', 'ayah_tr', 'reciter_id', 'reciter_name', 'audio']
sample ayah_ar: بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ
reciter column: reciter_id
ayahs to process: 4000


In [5]:
# Forced aligner: stable-ts gives word-level timestamps from whisper-medium.
# We feed the KNOWN verse text so it aligns rather than guesses.
#
# IMPORTANT: stable-ts/whisper loads audio via its own decoder and does NOT
# reliably accept a raw numpy array (that causes "Format not recognised").
# So we write each verse to a temp 16 kHz WAV with soundfile and pass the PATH.
import os
import tempfile
import soundfile as sf
import stable_whisper

torch.serialization.add_safe_globals(["whisper", "whisper.model.Whisper"])
aligner = stable_whisper.load_model("medium", device=DEVICE)

def align_words(audio_array, verse_text):
    """Return list of (word_text, start_s, end_s) for the verse."""
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
        tmp_path = tf.name
    try:
        sf.write(tmp_path, np.asarray(audio_array, dtype=np.float32), SAMPLING_RATE)
        result = aligner.align(tmp_path, verse_text, language="ar")
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
    out = []
    for seg in result.segments:
        for w in seg.words:
            out.append((w.word.strip(), float(w.start), float(w.end)))
    return out

_ex = ds[0]
_aligned = align_words(_ex["audio"]["array"], _ex["ayah_ar"])
print("smoke test OK:", len(_aligned), "words from verse 0")
print("first words:", _aligned[:5])

100%|██████████████████████████████████████| 1.42G/1.42G [00:07<00:00, 202MiB/s]
Align: 100%|██████████| 34.48/34.48 [00:07<00:00,  4.79sec/s]
Adjustment: 100%|██████████| 24.94/24.94 [00:00<00:00, 38107.81sec/s]

smoke test OK: 15 words from verse 0
first words: [('وَلْيَخْشَ', 0.08, 1.34), ('الَّذِينَ', 1.34, 2.2), ('لَوْ', 2.2, 2.58), ('تَرَكُوا', 2.58, 3.56), ('مِنْ', 3.56, 3.84)]


In [6]:
rows = {"audio_array": [], "labels": [], "text": [], "surah_id": [], "reciter": []}
n_words, n_dropped = 0, 0

for i in range(len(ds)):
    try:
        ex = ds[i]
        audio = ex["audio"]["array"]
    except Exception as e:
        print(f"Skipping bad audio at {i}: {e}")
        n_dropped += 1
        continue

    verse = ex["ayah_ar"]
    surah = ex.get("surah_id", -1)
    reciter = str(ex[RECITER_COL])

    try:
        aligned = align_words(audio, verse)
    except Exception:
        n_dropped += 1
        continue

    for word, start, end in aligned:
        cleaned = clean_quranic_text(word)
        if not cleaned:
            continue

        a = max(0, int((start - PAD_S) * SAMPLING_RATE))
        b = min(len(audio), int((end + PAD_S) * SAMPLING_RATE))
        clip = audio[a:b]

        if not (0.05 < len(clip) / SAMPLING_RATE < 30.0):
            continue

        labels = processor.tokenizer(cleaned).input_ids

        if not (0 < len(labels) <= MAX_LABEL_LEN):
            continue

        rows["audio_array"].append(np.asarray(clip, dtype=np.float32))
        rows["labels"].append(labels)
        rows["text"].append(cleaned)
        rows["surah_id"].append(int(surah))
        rows["reciter"].append(reciter)

        n_words += 1
        
print("word clips:", n_words)
print("dropped verses:", n_dropped)

Align: 100%|██████████| 34.48/34.48 [00:01<00:00, 19.30sec/s]
Adjustment: 100%|██████████| 24.94/24.94 [00:00<00:00, 30085.11sec/s]


Skipping bad audio at 1: Error opening <_io.BytesIO object at 0x7f52d7385d00>: Format not recognised.
Skipping bad audio at 2: Error opening <_io.BytesIO object at 0x7f52d7385fd0>: Format not recognised.


Align: 100%|██████████| 23.51/23.51 [00:01<00:00, 14.37sec/s]
Adjustment: 100%|██████████| 23.18/23.18 [00:00<00:00, 24197.10sec/s]
Align: 100%|██████████| 5.54/5.54 [00:00<00:00,  9.50sec/s]
Adjustment: 100%|██████████| 5.22/5.22 [00:00<00:00, 14152.73sec/s]
Align: 100%|██████████| 5.87/5.87 [00:00<00:00, 13.34sec/s]
Adjustment: 100%|██████████| 5.84/5.84 [00:00<00:00, 12066.37sec/s]
Align: 100%|██████████| 9.74/9.74 [00:00<00:00, 21.43sec/s]
Adjustment: 100%|██████████| 9.72/9.72 [00:00<00:00, 24209.40sec/s]
Align: 100%|██████████| 31.32/31.32 [00:00<00:00, 67.23sec/s]
Adjustment: 100%|██████████| 15.84/15.84 [00:00<00:00, 23459.67sec/s]
Align: 100%|██████████| 9.4/9.4 [00:01<00:00,  7.20sec/s] 
Adjustment: 100%|██████████| 6.72/6.72 [00:00<00:00, 34754.28sec/s]
Align: 100%|██████████| 13.58/13.58 [00:00<00:00, 28.96sec/s]
Adjustment: 100%|██████████| 13.22/13.22 [00:00<00:00, 21873.25sec/s]
Align: 100%|██████████| 7.13/7.13 [00:00<00:00,  8.06sec/s]
Adjustment: 100%|██████████| 7.12

Skipping bad audio at 43: Error opening <_io.BytesIO object at 0x7f52d73a21b0>: Format not recognised.


Align: 100%|██████████| 5.8/5.8 [00:00<00:00, 12.00sec/s] 
Adjustment: 100%|██████████| 5.68/5.68 [00:00<00:00, 17491.66sec/s]
[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (960) too large for available bit count (928)
Align: 100%|██████████| 17.03/17.03 [00:01<00:00, 16.99sec/s]
Adjustment: 100%|██████████| 16.82/16.82 [00:00<00:00, 23578.94sec/s]
Align: 100%|██████████| 8.28/8.28 [00:00<00:00, 16.56sec/s]
Adjustment: 100%|██████████| 8.12/8.12 [00:00<00:00, 20628.56sec/s]
Align: 100%|██████████| 6.77/6.77 [00:00<00:00, 13.77sec/s]
Adjustment: 100%|██████████| 6.64/6.64 [00:00<00:00, 13946.01sec/s]
Align: 100%|██████████| 81.14/81.14 [00:05<00:00, 14.12sec/s]
Adjustment: 100%|██████████| 79.38/79.38 [00:00<00:00, 40046.17sec/s]
Align: 100%|██████████| 27.43/27.43 [00:01<00:00, 17.61sec/s]
Adjustment: 100%|██████████| 27.26/27.26 [00:00<00:00, 31069.76sec/s]
Align: 100%|██████████| 19.49/19.49 [00:01<00:00, 18.70sec/s]
Adjustment: 100%|██████████| 19.1/19.1 [00

Skipping bad audio at 53: Error opening <_io.BytesIO object at 0x7f52d73b0310>: Format not recognised.


... (454 outputs left out)

Align: 100%|██████████| 10.84/10.84 [00:00<00:00, 19.73sec/s]
Adjustment: 100%|██████████| 9.58/9.58 [00:00<00:00, 19524.51sec/s]
[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (1024) too large for available bit count (928)
Align: 100%|██████████| 5.0/5.0 [00:00<00:00,  9.21sec/s] 
Adjustment: 100%|██████████| 4.98/4.98 [00:00<00:00, 20845.94sec/s]
Align: 100%|██████████| 12.12/12.12 [00:01<00:00,  7.28sec/s]
Adjustment: 100%|██████████| 10.98/10.98 [00:00<00:00, 21662.02sec/s]
Align: 100%|██████████| 29.62/29.62 [00:02<00:00, 10.15sec/s]
Adjustment: 100%|██████████| 28.5/28.5 [00:00<00:00, 32633.81sec/s]
Align: 100%|██████████| 123.08/123.08 [00:17<00:00,  6.93sec/s]
Adjustment: 100%|██████████| 122.54/122.54 [00:00<00:00, 48419.22sec/s]
Align: 100%|██████████| 37.73/37.73 [00:04<00:00,  8.03sec/s]
Adjustment: 100%|██████████| 36.66/36.66 [00:00<00:00, 37266.89sec/s]


Skipping bad audio at 3989: Error opening <_io.BytesIO object at 0x7f4a46155490>: Format not recognised.


Align: 100%|██████████| 5.96/5.96 [00:01<00:00,  5.44sec/s]
Adjustment: 100%|██████████| 5.8/5.8 [00:00<00:00, 19109.95sec/s]
Align: 100%|██████████| 5.17/5.17 [00:00<00:00,  9.32sec/s]
Adjustment: 100%|██████████| 4.98/4.98 [00:00<00:00, 15823.97sec/s]
Align: 100%|██████████| 3.81/3.81 [00:00<00:00,  6.97sec/s]
Adjustment: 100%|██████████| 3.8/3.8 [00:00<00:00, 12884.68sec/s]
Align: 100%|██████████| 19.17/19.17 [00:01<00:00, 16.96sec/s]
Adjustment: 100%|██████████| 18.54/18.54 [00:00<00:00, 21457.61sec/s]
Align: 100%|██████████| 8.46/8.46 [00:00<00:00, 15.09sec/s]
Adjustment: 100%|██████████| 8.44/8.44 [00:00<00:00, 19302.03sec/s]
Align: 100%|██████████| 17.5/17.5 [00:02<00:00,  6.34sec/s] 
Adjustment: 100%|██████████| 16.72/16.72 [00:00<00:00, 20076.94sec/s]
Align: 100%|██████████| 7.78/7.78 [00:00<00:00, 14.30sec/s]
Adjustment: 100%|██████████| 7.66/7.66 [00:00<00:00, 23230.92sec/s]
Align: 100%|██████████| 7.6/7.6 [00:02<00:00,  3.51sec/s]
Adjustment: 100%|██████████| 6.6/6.6 [00:00

word clips: 47153
dropped verses: 254


In [7]:
# SANITY CHECK -- inspect a few clips BEFORE trusting the whole batch.
# Bad word boundaries (clipped final harakah) poison training. Listen to these.
import IPython.display as ipd
for k in range(0, min(5, len(rows["text"]))):
    print("text:", rows["text"][k], "| dur:",
          round(len(rows["audio_array"][k]) / SAMPLING_RATE, 2), "s",
          "| reciter:", rows["reciter"][k])
    ipd.display(ipd.Audio(rows["audio_array"][k], rate=SAMPLING_RATE))

text: وَلْيَخْشَ | dur: 1.44 s | reciter: minshawy_teacher


text: الَّذِينَ | dur: 1.06 s | reciter: minshawy_teacher


text: لَوْ | dur: 0.58 s | reciter: minshawy_teacher


text: تَرَكُوا | dur: 1.18 s | reciter: minshawy_teacher


text: مِنْ | dur: 0.48 s | reciter: minshawy_teacher


In [8]:
audio_gb = sum(
    arr.nbytes for arr in rows["audio_array"]
) / (1024**3)

print(f"Audio RAM: {audio_gb:.2f} GB")

Audio RAM: 3.87 GB


In [9]:
from datasets import Dataset, concatenate_datasets
import gc

CHUNK_SIZE = 5000

datasets = []

for start in range(0, len(rows["text"]), CHUNK_SIZE):

    end = start + CHUNK_SIZE

    chunk = {
        k: v[start:end]
        for k, v in rows.items()
    }

    print(
        f"Building chunk "
        f"{start//CHUNK_SIZE + 1} "
        f"of "
        f"{(len(rows['text']) - 1)//CHUNK_SIZE + 1}"
    )

    datasets.append(
        Dataset.from_dict(chunk)
    )

    del chunk
    gc.collect()

ayah_words = concatenate_datasets(datasets)

del datasets
gc.collect()

ayah_words.save_to_disk(OUTPUT_DIR)

print(
    f"Saved {len(ayah_words)} "
    f"multi-voice word clips to "
    f"{OUTPUT_DIR}"
)

Building chunk 1 of 10
Building chunk 2 of 10
Building chunk 3 of 10
Building chunk 4 of 10
Building chunk 5 of 10
Building chunk 6 of 10
Building chunk 7 of 10
Building chunk 8 of 10
Building chunk 9 of 10
Building chunk 10 of 10


Saving the dataset (0/9 shards):   0%|          | 0/47153 [00:00<?, ? examples/s]

Saved 47153 multi-voice word clips to /kaggle/working/processed_quran_ayah_words
